# SHS generation + analysis

In [ ]:
# Sequence & structure 
RNA_SEQ =   "AUGCUACGAUCAGCUGAUCGAUGCGCGAUCGU"
STRUCTURE = "(((....((((....()()()()..)))))))"  # overrides the predictor if set
INTERACTIONS = None # "[[0, 1, 1], [2, 3, 1], [2, 4, 1], [5, 6, 1], [6, 7, 1], [8, 10, 0.9], [9, 10, 0.8]]"
MUTATION_RATES = None
STRUCTURE_PREDICTOR = "rnafold"          # rnafold, spotrna, rnaformer

# MSA size / reproducibility 
N = 10000
SEED = 42

# Mutation parameters
MUTATION_RATE_UNPAIRED = 0.1
MUTATION_RATE_PAIRED   = 0.2
PAIR_MUTATION_APPROACH = "covariance" # covariance | watson_crick | watson_crick_cov | original | none

# Triplet Mutation parameter for "original" mutation approach
WOBBLE_PROB            = 0.0 # 0.1 

# insertions / deletions - general
MAX_INSERTION_FRACTION = 0.2 # 0.1
MAX_DELETION_FRACTION  = 0.1 # 0.1
# paired
STEM_SINGLE_INSERTION_PROB  = 0.05
STEM_LONG_INSERTION_PROB    = 0.01 # 0.05
STEM_SINGLE_DELETION_PROB   = 0.1
STEM_PAIR_DELETION_PROB     = 0.0

# unpaired
LOOP_SINGLE_INSERTION_PROB = 0.8 # 0.2
LOOP_SINGLE_DELETION_PROB  = 0.3 # 0.8
LOOP_LONG_INSERTION_PROB   = 0.0 # 0.05
LOOP_LONG_DELETION_PROB    = 0.05 # 0.05

# Output
OUTPUT_JSON_DIR = "custom_msa_json_output"
PDB_ID = "None"

## Setup

In [ ]:
import importlib
import os
import sys
from argparse import Namespace
from pathlib import Path

%matplotlib inline

_candidates = [Path.cwd(), Path.cwd() / "SHS-Generator"]
SHS_DIR = next((p for p in _candidates if (p / "shs_generator.py").exists()), None)
if SHS_DIR is None:
    raise FileNotFoundError("Could not find shs_generator.py.")
SHS_DIR = SHS_DIR.resolve()
os.chdir(SHS_DIR)
if str(SHS_DIR) not in sys.path:
    sys.path.insert(0, str(SHS_DIR))

import pair_map
import shs_generator
import shs_analyzer
importlib.reload(pair_map)
importlib.reload(shs_generator)
importlib.reload(shs_analyzer)

print("SHS-Generator dir:", SHS_DIR)

## Generate the SHS

In [ ]:

STRUCTURE = "".join(["\"" + c + "\"" if c.isdigit() else c for c in STRUCTURE])
import random
# r = lambda x = 1: random.randint(0, len(RNA_SEQ))
# INTERACTIONS = str([[r(), r(), random.random()] for _ in range(len(RNA_SEQ) // 2)])

args = Namespace(
    structure_predictor        = None if STRUCTURE or INTERACTIONS else STRUCTURE_PREDICTOR,
    structure                  = None if INTERACTIONS else STRUCTURE,
    interactions               = INTERACTIONS,
    mutation_rates             = MUTATION_RATES,
    rna_seq                    = RNA_SEQ,
    protein_seq                = "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQ",  # placeholder; AF3 JSON needs a protein chain
    input_json_path            = None,
    pdb_id                     = PDB_ID,
    output_json_dir            = OUTPUT_JSON_DIR,
    N                          = N,
    mutation_rate_unpaired     = MUTATION_RATE_UNPAIRED,
    mutation_rate_paired       = MUTATION_RATE_PAIRED,
    pair_mutation_approach     = PAIR_MUTATION_APPROACH,
    stem_single_insertion_prob = STEM_SINGLE_INSERTION_PROB,
    stem_long_insertion_prob   = STEM_LONG_INSERTION_PROB,
    stem_single_deletion_prob  = STEM_SINGLE_DELETION_PROB,
    stem_pair_deletion_prob    = STEM_PAIR_DELETION_PROB,
    loop_single_insertion_prob = LOOP_SINGLE_INSERTION_PROB,
    loop_single_deletion_prob  = LOOP_SINGLE_DELETION_PROB,
    loop_long_insertion_prob   = LOOP_LONG_INSERTION_PROB,
    loop_long_deletion_prob    = LOOP_LONG_DELETION_PROB,
    max_insertion_fraction     = MAX_INSERTION_FRACTION,
    max_deletion_fraction      = MAX_DELETION_FRACTION,
    wobble_prob                = WOBBLE_PROB,
    seed                       = SEED,
    max_chains                 = None,
    plot                       = False,
    print_msa                  = False,
    show_plot                  = False,
)

generator = shs_generator.MsaGenerator(args)
generated_json = generator.process(write=False)
print("Generated AF3 JSON with", len(generated_json["sequences"]), "chains; name:", generated_json["name"])

## Analyze & render

In [ ]:
import json
import numpy as np

data = shs_analyzer.MsaData.from_text(json.dumps(generated_json), pairs=generator.pair_map)
feat = shs_analyzer.Features(data)

print(f"Query length: {len(data.seq)}  "
   f"|  MSA rows: {data.aligned.shape[0] + 1}  "
   f"|  covariance shape: {feat.covariance.shape}")

mat = data.pairs._pairs_mat.copy()
mat[~np.eye(mat.shape[0], dtype=bool)] *= 0.15
vmin, vmax = min(mat.min(), feat.covariance.min()), max(mat.max(), feat.covariance.max())
shs_analyzer.plot_covariance(data.seq, feat.covariance, show_values=False, vmin=vmin, vmax=vmax)
shs_analyzer.plot_covariance(data.seq, mat, title="Input interaction strength and mutation rates", show_values=False, vmin=vmin, vmax=vmax)
shs_analyzer.plot_covariance(data.seq, np.abs(mat - feat.covariance), title="Error", show_values=False, vmin=vmin, vmax=vmax)
shs_analyzer.plot_covariance_classification(data.seq, feat.covariance, pairs=data.pairs)
shs_analyzer.plot_deletions_and_insertion(feat.col_deletion_rate, feat.col_insertion_mean, pairs=data.pairs)

print(data.seq)
for row in data.raw[:10]:
    print("".join(row))

## Recovered parameters

In [ ]:
import pandas as pd

# Show full cell contents so the `detail` strings aren't truncated.
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

# Known generator input per recovered parameter name (the ground truth each
# estimator tries to reproduce). Extend this as you add estimators.
KNOWN_INPUTS = {
    "n_sequences":            N,
    "mutation_rate_unpaired": MUTATION_RATE_UNPAIRED,
    "mutation_rate_paired":   MUTATION_RATE_PAIRED,
}

report = shs_analyzer.analyze_data(data)


def _fmt_detail(detail):
    # Turn {'unpaired_cols': 14, ...} into a readable "unpaired_cols=14 · ..." line.
    return " · ".join(f"{k}={v}" for k, v in detail.items())


rows = []
for name, est in report.items():
    known = KNOWN_INPUTS.get(name)
    # est.value != est.value is True only for NaN -> no error there.
    error = abs(est.value - known) if (known is not None and est.value == est.value) else None
    rows.append({
        "parameter": name,
        "known":     known,
        "recovered": round(est.value, 4),
        "abs_error": round(error, 4) if error is not None else None,
        "detail":    _fmt_detail(est.detail),
    })

pd.DataFrame(rows).set_index("parameter")